In [ ]:
!pip install git+https://github.com/openai/CLIP.git

!git clone https://github.com/victoriachernova/styleclip-nada.git
%cd styleclip-nada

!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git /content/stylegan2-ada-pytorch

!mkdir -p weights
!wget  -q -O weights/ffhq.pkl https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl
!ls -lh weights/ffhq.pkl

!pip install -q ninja
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
import os
import sys
sys.path.insert(0,'..')
import clip
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import torchvision.utils as vutils
import torch.nn.functional as F

import importlib
import src.training.loss
importlib.reload(src.training.loss)
import src.training.trainer
importlib.reload(src.training.trainer)
import src.models.mapper
importlib.reload(src.models.mapper)

from src.models.generator import Generator
from src.models.mapper import Mapper
from src.models.clip_model import CLIP
from src.training.loss import DirectCLIPloss
from src.training.trainer import MapperTrainer
from src.utils import show_grid, save_checkpoint, set_seed, load_checkpoint

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 42

weights_path = 'weights/ffhq.pkl'
target_text = 'a photo of a person with sketch style'

n_iters = 300
batch_size = 4
log_every = 25
lr = 1e-4

checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
set_seed(seed)

In [ ]:
G = Generator('weights/ffhq.pkl', device=device)
mapper = Mapper().to(device)
loss_fn = DirectCLIPloss()
trainer = MapperTrainer(mapper, G, loss_fn, device=device)

print(f'Generator z_dim: {G.G.z_dim}, num_ws: {G.G.num_ws}')
print(f'Mapper paramerters: {sum(p.numel() for p in mapper.parameters()):,}')

In [ ]:
for name, params in G.G.named_parameters():
  print(name, params.shape)

In [ ]:
torch.manual_seed(seed)
z_test = torch.randn(4, G.G.z_dim, device=device)
with torch.no_grad():
  w_test = G.mapping(z_test)
  img_base = G.synthesis(w_test)

show_grid(img_base, title='Baseline')

In [ ]:
source_text = 'a photo of a person'

styles = [
    'an anime portrait',
    'a photo of a person as a zombie',
    'a realistic vampire'
]
results = {}
histories = {}

for style in styles:

  np.random.seed(seed)

  torch.manual_seed(seed)

  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

  mapper_i = Mapper().to(device)
  trainer_i = MapperTrainer(mapper_i, G, loss_fn, lr=1e-4, device=device)

  for p in G.G.parameters():
    p.requires_grad = False

  for name, p in G.G.named_parameters():
    if "synthesis.b1024" in name:
      p.requires_grad = True

  trainer_i.optimizer = torch.optim.Adam(
      list(trainer_i.mapper.parameters()) +
      list(filter(lambda p: p.requires_grad, trainer_i.generator.G.parameters())),
      lr=1e-4
    )

  history = []

  for i in range(n_iters):
    metrics = trainer_i.train_step(source_text, style)
    history.append(metrics)

  with torch.no_grad():
    delta = mapper_i(w_test)
    img_i = G.synthesis(w_test + delta)

  results[style] = img_i.detach().cpu()
  histories[style] = history

  style_name = style.replace(" ", "_").replace("/", "_")
  ckpt_name = f"{checkpoint_dir}/mapper_{style_name}.pt"
  save_checkpoint(mapper_i, ckpt_name, extra={'target_text': style, 'n_iters': 300})

  del delta
  del img_i
  del trainer_i
  del mapper_i
  torch.cuda.empty_cache()

In [ ]:
print(os.listdir(checkpoint_dir))

In [ ]:
style = "an anime portrait"

history = histories[style]

totals = [h['total'] for h in history]
clips = [h['clip'] for h in history]
l2s = [h['l2'] for h in history]

plt.figure(figsize=(10,4))
plt.plot(totals, label='total')
plt.plot(clips, label='directional clip')
plt.plot(l2s, label='l2')

plt.xlabel('iteration')
plt.ylabel('loss')
plt.legend()
plt.title(f"training: {style}")
plt.show()

In [ ]:
style = "a photo of a person as a zombie"

history = histories[style]

totals = [h['total'] for h in history]
clips = [h['clip'] for h in history]
l2s = [h['l2'] for h in history]

plt.figure(figsize=(10,4))
plt.plot(totals, label='total')
plt.plot(clips, label='directional clip')
plt.plot(l2s, label='l2')

plt.xlabel('iteration')
plt.ylabel('loss')
plt.legend()
plt.title(f"training: {style}")
plt.show()

In [ ]:
style = "a realistic vampire"

history = histories[style]

totals = [h['total'] for h in history]
clips = [h['clip'] for h in history]
l2s = [h['l2'] for h in history]

plt.figure(figsize=(10,4))
plt.plot(totals, label='total')
plt.plot(clips, label='directional clip')
plt.plot(l2s, label='l2')

plt.xlabel('iteration')
plt.ylabel('loss')
plt.legend()
plt.title(f"training: {style}")
plt.show()

In [ ]:
before = F.interpolate(img_base.cpu(), size=256)
anime = F.interpolate(results["an anime portrait"], size=256)
zombie = F.interpolate(results["a photo of a person as a zombie"], size=256)
vampire = F.interpolate(results["a realistic vampire"], size=256)

fig, axes = plt.subplots(1, 4, figsize=(20,5))

images = [before, anime, zombie, vampire]
titles = ["Before", "Anime",  "Zombie", "Vampire"]

for ax, imgs, title in zip(axes, images, titles):

    grid = vutils.make_grid((imgs.clamp(-1,1)+1)/2, nrow=2)

    ax.imshow(grid.permute(1,2,0))
    ax.set_title(title, fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login, HfApi

login()

api = HfApi()
api.create_repo('tvictoria/styleclip-nada-ffhq', repo_type='model', exist_ok=True)

In [ ]:
for file in os.listdir(checkpoint_dir):

    if file.endswith(".pt"):

      path = os.path.join(checkpoint_dir, file)

      api.upload_file(path_or_fileobj=path, path_in_repo=file, repo_id="tvictoria/styleclip-nada-ffhq")

      print(f"Uploaded {file}")

In [ ]:
os.makedirs(
    "../checkpoints",
    exist_ok=True
)

In [ ]:
torch.save(
    mapper.state_dict(),
    "../checkpoints/slyleclip.pt"
)